# NB03 — Physical Properties and Data Quality
**TCC Weathering ML — Schema v2.2**

The oil as a physical material. This notebook characterizes the ECCC ESTS
oils through their measurable properties — density, viscosity, pour point,
flash point, surface tension, water content, adhesion — and how these change
under evaporative weathering (W0 → W3).

**Why this matters for the pipeline:**
- Physical properties are NOT features in the ML model (D25: macroscopic
  properties excluded). But they contextualize model performance: oils with
  extreme pour point or high wax may behave differently in LOOO.
- NB05 correlates these with MAE.
- Derived properties (Ea, spreading coefficient) feed the Dissertation and LGAF.

**Depends on:** NB00 (schema), NB01 (sample_properties, oils, weathering_stages)
**Produces:** ~6 derived properties in `sample_properties`, ~10 figures,
outlier inventory, data richness heatmap, literature comparison table.


## 1. Setup

Import shared infrastructure and configure figure output directory.


In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from IPython.display import Image, display

from utils import (
    get_conn, SEED, FIG_ROOT,
    setup_figure_style,
    STAGE_COLORS, OILTYPE_COLORS,
)

FIG_DIR = FIG_ROOT / 'nb03'
FIG_DIR.mkdir(parents=True, exist_ok=True)
setup_figure_style()
import matplotlib as _mpl
_mpl.rcParams['font.family'] = 'DejaVu Sans'  # full Unicode support
np.random.seed(SEED)

STAGES = ['W0', 'W1', 'W2', 'W3']


## 2. Sanity checks

Verify NB01 outputs before any analysis. Every check has an expected value;
if any fails, the notebook stops. This prevents silent corruption from upstream.


In [ ]:
with get_conn() as conn:
    checks = {}
    checks['n_oils_included'] = conn.execute(
        'SELECT COUNT(*) FROM oils WHERE include_in_analysis=1'
    ).fetchone()[0]
    checks['n_oils_total'] = conn.execute(
        'SELECT COUNT(*) FROM oils'
    ).fetchone()[0]
    checks['n_compounds'] = conn.execute(
        'SELECT COUNT(*) FROM compounds WHERE excluded=0'
    ).fetchone()[0]
    checks['n_measurements'] = conn.execute(
        'SELECT COUNT(*) FROM measurements'
    ).fetchone()[0]
    checks['n_sample_props'] = conn.execute(
        'SELECT COUNT(*) FROM sample_properties'
    ).fetchone()[0]
    checks['n_pan_evap'] = conn.execute(
        'SELECT COUNT(*) FROM pan_evaporation'
    ).fetchone()[0]
    
    # D15 check
    checks['d15_violation'] = conn.execute(
        'SELECT COUNT(*) FROM measurements WHERE is_imputed=1 AND value_imputed > 0'
    ).fetchone()[0]
    
    # D14 check
    checks['d14_residual'] = conn.execute('''
        SELECT COUNT(*) FROM (
            SELECT m.oil_id, m.compound_id
            FROM measurements m
            JOIN oils o ON m.oil_id = o.oil_id
            JOIN compounds c ON m.compound_id = c.compound_id
            WHERE o.include_in_analysis = 1 AND c.excluded = 0
            GROUP BY m.oil_id, m.compound_id
            HAVING SUM(CASE WHEN value_imputed = 0 THEN 1 ELSE 0 END) = 4
               AND COUNT(*) = 4
        )
    ''').fetchone()[0]
    
    # CSV hash
    csv_hash = conn.execute(
        "SELECT value FROM db_metadata WHERE key='csv_sha256'"
    ).fetchone()
    checks['csv_hash'] = csv_hash[0][:16] + '...' if csv_hash else 'NOT FOUND'
    
    # Property types
    prop_types = conn.execute('''
        SELECT property_name, COUNT(*) as n
        FROM sample_properties
        GROUP BY property_name
        ORDER BY n DESC
    ''').fetchall()
    checks['n_property_types'] = len(prop_types)

print('Database sanity checks:')
for k, v in checks.items():
    print(f'  {k:25s}: {v}')

assert checks['n_oils_included'] >= 40, f'Too few included oils: {checks["n_oils_included"]}'
assert checks['n_sample_props'] > 10000, f'Too few sample_properties: {checks["n_sample_props"]}'
assert checks['d15_violation'] == 0, f'D15 violated: {checks["d15_violation"]} imputed > 0'
assert checks['d14_residual'] == 0, f'D14 not applied: {checks["d14_residual"]} uniform zeros'
print('\n✓ All checks passed')

# Show property type inventory
print(f'\nProperty types in database ({checks["n_property_types"]}):')
for pname, n in prop_types[:30]:
    print(f'  {pname:55s}: {n:>5,} records')
if len(prop_types) > 30:
    print(f'  ... and {len(prop_types)-30} more')


## 2b. Property name inventory

List all distinct property names and conditions in the database.
This ensures NB03 queries match what NB01 actually stored.
If a query returns empty, check this inventory first.


In [ ]:
with get_conn() as conn:
    prop_inventory = pd.read_sql('''
        SELECT property_name, condition, temperature_c,
               COUNT(*) as n_records,
               COUNT(DISTINCT oil_id) as n_oils
        FROM sample_properties
        WHERE oil_id IN (SELECT oil_id FROM oils WHERE include_in_analysis=1)
        GROUP BY property_name, condition, temperature_c
        ORDER BY property_name, condition, temperature_c
    ''', conn)

print(f'Property inventory: {len(prop_inventory)} unique (name, condition, temperature) combinations')
print(f'Unique property names: {prop_inventory["property_name"].nunique()}')
print()

# Show tension-related (critical for C1)
tension = prop_inventory[prop_inventory['property_name'].str.contains('tension', case=False)]
if len(tension) > 0:
    print('Tension properties (critical — check names match NB03 queries):')
    for _, row in tension.iterrows():
        print(f'  {row["property_name"]:45s} cond={str(row["condition"]):15s} '
              f'T={row["temperature_c"]}  n={row["n_records"]}')
else:
    print('⚠️ No tension properties found!')

# Show viscosity-related (critical for I5)
visc = prop_inventory[prop_inventory['property_name'].str.contains('viscosity|visc', case=False)]
if len(visc) > 0:
    print(f'\nViscosity properties ({len(visc)} combinations):')
    for _, row in visc.iterrows():
        print(f'  {row["property_name"]:45s} cond={str(row["condition"]):15s} '
              f'T={row["temperature_c"]}  n={row["n_records"]}')

# Store for use by other cells
PROP_NAMES = set(prop_inventory['property_name'])
print(f'\n✓ PROP_NAMES set created with {len(PROP_NAMES)} unique names')


## 3. Load oil metadata

Base table for all downstream analyses: included oils with their type,
API, density, pour point. This is the reference frame.


In [ ]:
with get_conn() as conn:
    df_oils = pd.read_sql('''
        SELECT oil_id, oil_name, oil_type, include_in_analysis,
               api_gravity, density_15c, pour_point_c, flash_point_c,
               sulfur_pct, wax_pct
        FROM oils
        ORDER BY oil_type, oil_name
    ''', conn)

df_incl = df_oils[df_oils['include_in_analysis'] == 1].copy()
print(f'Included oils: {len(df_incl)} of {len(df_oils)}')
print(f'\nOil types:')
for otype, sub in df_incl.groupby('oil_type'):
    print(f'  {otype:20s}: {len(sub)} oils')

# Helper to load a property across stages
def load_prop_by_stage(conn, prop_name, temperature_c=None, condition=None):
    """Load a specific property for all included oils across W0-W3."""
    query = '''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code, sp.value, sp.std_dev
        FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = ?
          AND o.include_in_analysis = 1
          AND sp.stage_code IN ('W0','W1','W2','W3')
    '''
    params = [prop_name]
    if temperature_c is not None:
        query += ' AND sp.temperature_c = ?'
        params.append(temperature_c)
    if condition is not None:
        query += ' AND sp.condition = ?'
        params.append(condition)
    return pd.read_sql(query, conn, params=params)


## 4. Density (3 temperatures)

Density measured at 0°C, 5°C, and 15°C (Anton Paar DMA 5000 or SVM 3000).
Mean of triplicates. ASTM D5002.

**Expected behavior:** density decreases with temperature (thermal expansion).
Density increases with weathering (light volatiles evaporate, heavy residue
concentrates). Crude oils: 0.78-0.98 g/mL. Bitumen: >1.0 g/mL.

**Derived property:** thermal expansion coefficient
β = −(1/ρ)(dρ/dT), estimated from linear regression on 3 temperatures.
**Caveat:** only 3 points (0, 5, 15°C) → 1 degree of freedom. R² is
almost always high. The derived β is indicative, not precise.


In [ ]:
with get_conn() as conn:
    temps = [0.0, 5.0, 15.0]
    df_dens = pd.concat([
        load_prop_by_stage(conn, 'density', temperature_c=t).assign(temp_c=t)
        for t in temps
    ])

if len(df_dens) > 0:
    print(f'Density records: {len(df_dens)} ({df_dens["oil_id"].nunique()} oils)')
    
    # ── Fig 1: Density vs temperature, colored by stage ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: ρ(T) curves for W0, colored by oil_type
    ax = axes[0]
    w0_dens = df_dens[df_dens['stage_code'] == 'W0']
    for otype in sorted(w0_dens['oil_type'].unique()):
        sub = w0_dens[w0_dens['oil_type'] == otype]
        for oil_id in sub['oil_id'].unique():
            oil_data = sub[sub['oil_id'] == oil_id].sort_values('temp_c')
            ax.plot(oil_data['temp_c'], oil_data['value'],
                    color=OILTYPE_COLORS.get(otype, 'grey'), alpha=0.4, linewidth=0.8)
    # Legend via proxy artists
    for otype in sorted(w0_dens['oil_type'].unique()):
        ax.plot([], [], color=OILTYPE_COLORS.get(otype, 'grey'),
                linewidth=2, label=otype)
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Density (g/mL)')
    ax.set_title('W0 Density vs Temperature by Oil Type')
    ax.legend(fontsize=7)
    
    # Right: Density at 15°C, W0 vs W3
    ax = axes[1]
    d15 = df_dens[df_dens['temp_c'] == 15.0]
    w0_d15 = d15[d15['stage_code'] == 'W0'].groupby('oil_id')['value'].mean()
    w3_d15 = d15[d15['stage_code'] == 'W3'].groupby('oil_id')['value'].mean()
    common = w0_d15.index.intersection(w3_d15.index)
    ax.scatter(w0_d15.loc[common], w3_d15.loc[common], s=30, alpha=0.7)
    lims = [min(w0_d15.loc[common].min(), w3_d15.loc[common].min()) - 0.02,
            max(w0_d15.loc[common].max(), w3_d15.loc[common].max()) + 0.02]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='1:1 line')
    ax.set_xlabel('Density W0 (g/mL)')
    ax.set_ylabel('Density W3 (g/mL)')
    ax.set_title('Density Shift with Weathering (15°C)')
    ax.legend()
    
    plt.tight_layout()
    fig_path = FIG_DIR / 'nb03_density.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    
    # ── Derive thermal expansion coefficient β ──
    beta_records = []
    for oil_id in w0_dens['oil_id'].unique():
        oil_data = w0_dens[w0_dens['oil_id'] == oil_id].sort_values('temp_c')
        if len(oil_data) >= 3:
            T = oil_data['temp_c'].values
            rho = oil_data['value'].values
            if all(rho > 0):
                slope, _, r, _, _ = stats.linregress(T, rho)
                if r**2 < 0.95:  # reject poor linear fit
                    continue
                rho_mean = rho.mean()
                beta = -slope / rho_mean  # 1/°C
                beta_records.append((
                    int(oil_id), 'W0', 'thermal_expansion_coeff',
                    None, None, float(beta), '1/°C',
                    None, None, 'computed'
                ))
    
    with get_conn() as conn:
        conn.execute("DELETE FROM sample_properties WHERE property_name='thermal_expansion_coeff'")
        conn.executemany(
            '''INSERT OR IGNORE INTO sample_properties
               (oil_id, stage_code, property_name, temperature_c, condition,
                value, unit, std_dev, replicates, method)
               VALUES (?,?,?,?,?,?,?,?,?,?)''',
            beta_records
        )
    print(f'\n✓ {len(beta_records)} thermal expansion coefficients persisted')
    if beta_records:
        vals = [r[5] for r in beta_records]
        print(f'  β range: {min(vals)*1e4:.2f} – {max(vals)*1e4:.2f} × 10⁻⁴ /°C')
else:
    print('⚠️ No density data found')


## 5. API gravity

Computed from density at 0°C and 15°C using ECCC's exponential interpolation
(ASTM D5002). Not the simplified 141.5/ρ₁₅ − 131.5.

Light oils: API > 31. Medium: 22-31. Heavy: 10-22. Extra-heavy: < 10.


In [ ]:
if 'api_gravity' in df_incl.columns and df_incl['api_gravity'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: API distribution by oil_type
    ax = axes[0]
    for otype in sorted(df_incl['oil_type'].unique()):
        sub = df_incl[df_incl['oil_type'] == otype]['api_gravity'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=10, alpha=0.5, label=f'{otype} (n={len(sub)})',
                    color=OILTYPE_COLORS.get(otype, 'grey'))
    ax.set_xlabel('API Gravity')
    ax.set_ylabel('Count')
    ax.set_title('API Gravity Distribution by Oil Type')
    ax.axvline(31, color='grey', ls=':', alpha=0.5)
    ax.axvline(22, color='grey', ls=':', alpha=0.5)
    ax.axvline(10, color='grey', ls=':', alpha=0.5)
    ax.text(35, ax.get_ylim()[1]*0.9, 'Light', fontsize=8, color='grey')
    ax.text(24, ax.get_ylim()[1]*0.9, 'Medium', fontsize=8, color='grey')
    ax.text(12, ax.get_ylim()[1]*0.9, 'Heavy', fontsize=8, color='grey')
    ax.legend(fontsize=7)
    
    # Right: API vs density at 15°C
    ax = axes[1]
    valid = df_incl.dropna(subset=['api_gravity', 'density_15c'])
    for otype in sorted(valid['oil_type'].unique()):
        sub = valid[valid['oil_type'] == otype]
        ax.scatter(sub['density_15c'], sub['api_gravity'], s=40, alpha=0.7,
                   color=OILTYPE_COLORS.get(otype, 'grey'), label=otype)
    x = np.linspace(0.7, 1.1, 100)
    ax.plot(x, 141.5/x - 131.5, 'k--', alpha=0.4, label='Simplified formula')
    ax.set_xlabel('Density at 15°C (g/mL)')
    ax.set_ylabel('API Gravity')
    ax.set_title('API vs Density (W0)')
    ax.legend(fontsize=7)
    
    plt.tight_layout()
    fig_path = FIG_DIR / 'nb03_api_gravity.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    
    # Summary stats
    print('API Gravity summary (W0, included oils):')
    for otype in sorted(df_incl['oil_type'].unique()):
        sub = df_incl[df_incl['oil_type'] == otype]['api_gravity'].dropna()
        if len(sub) > 0:
            print(f'  {otype:20s}: {sub.median():5.1f} [{sub.min():.1f} – {sub.max():.1f}] (n={len(sub)})')
else:
    print('⚠️ No API gravity data')


## 6. Viscosity: rheology and Arrhenius activation energy

Dynamic viscosity measured at 3 temperatures × 6 shear conditions.
ECCC used multiple instruments (HAAKE RotoVisco, RS300/6000, Stabinger SVM 3000,
VTiQ) for different viscosity ranges (ASTM D7042, 12.05/x.x/M, 12.06/x.x/M).

**Newtonian behavior:** viscosity independent of shear rate. Expected for light
crude and refined products.
**Non-Newtonian:** viscosity varies with shear (shear-thinning). Expected for
heavy crudes, bitumen, weathered oils. Indicates internal structure (wax, asphaltenes).

**Derived properties:**
- Arrhenius Ea: ln(η) = ln(A) + Ea/(RT). Higher Ea = more temperature-sensitive.
- Kinematic viscosity: ν = η/ρ (at matching temperature).
- Pseudoplasticity index: ratio η(shear=0.1)/η(shear=100).


In [ ]:
with get_conn() as conn:
    df_visc = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code,
               sp.temperature_c, sp.condition, sp.value
        FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'dynamic_viscosity'
          AND o.include_in_analysis = 1
          AND sp.stage_code IN ('W0','W1','W2','W3')
          AND sp.value > 0
    ''', conn)

if len(df_visc) > 0:
    print(f'Viscosity records: {len(df_visc)}')
    print(f'  Oils: {df_visc["oil_id"].nunique()}')
    print(f'  Conditions: {sorted(df_visc["condition"].unique())}')
    print(f'  Temperatures: {sorted(df_visc["temperature_c"].dropna().unique())}')
    
    # Newtonian vs non-Newtonian census
    newt = df_visc[df_visc['condition'] == 'newtonian']
    non_newt = df_visc[df_visc['condition'] != 'newtonian']
    print(f'\n  Newtonian records: {len(newt)} ({newt["oil_id"].nunique()} oils)')
    print(f'  Non-Newtonian records: {len(non_newt)} ({non_newt["oil_id"].nunique()} oils)')
    
    # ── Arrhenius Ea (Newtonian W0 only) ──
    R_GAS = 8.314  # J/(mol·K)
    ea_records = []
    w0_newt = newt[(newt['stage_code'] == 'W0') & newt['temperature_c'].notna()]
    
    for oil_id in w0_newt['oil_id'].unique():
        sub = w0_newt[w0_newt['oil_id'] == oil_id].sort_values('temperature_c')
        if len(sub) < 3:
            continue
        T_K = sub['temperature_c'].values + 273.15
        ln_eta = np.log(sub['value'].values)
        slope, intercept, r_val, p_val, se = stats.linregress(1.0 / T_K, ln_eta)
        Ea = slope * R_GAS / 1000  # kJ/mol
        
        if r_val**2 > 0.80 and Ea > 0:
            ea_records.append({
                'oil_id': int(oil_id), 'Ea_kJmol': float(Ea),
                'r2': float(r_val**2), 'n_points': len(sub)
            })
    
    df_ea = pd.DataFrame(ea_records)
    if len(df_ea) > 0:
        print(f'\n  Arrhenius Ea computed for {len(df_ea)} oils (R² > 0.80)')
        print(f'    Range: {df_ea["Ea_kJmol"].min():.1f} – {df_ea["Ea_kJmol"].max():.1f} kJ/mol')
        print(f'    Median: {df_ea["Ea_kJmol"].median():.1f} kJ/mol')
        
        # Persist
        with get_conn() as conn:
            conn.execute("DELETE FROM sample_properties WHERE property_name='viscosity_Ea_kJmol'")
            ea_persist = [
                (r['oil_id'], 'W0', 'viscosity_Ea_kJmol', None, None,
                 r['Ea_kJmol'], 'kJ/mol', None, None, 'computed')
                for _, r in df_ea.iterrows()
            ]
            conn.executemany(
                '''INSERT OR IGNORE INTO sample_properties
                   (oil_id, stage_code, property_name, temperature_c, condition,
                    value, unit, std_dev, replicates, method)
                   VALUES (?,?,?,?,?,?,?,?,?,?)''',
                ea_persist
            )
        print(f'    ✓ Persisted to sample_properties')
    
    # ── Kinematic viscosity at 15°C ──
    kin_records = []
    for stage in STAGES:
        visc_15 = newt[(newt['stage_code'] == stage) & (newt['temperature_c'] == 15.0)]
        dens_15 = df_dens[(df_dens['stage_code'] == stage) & (df_dens['temp_c'] == 15.0)] if len(df_dens) > 0 else pd.DataFrame()
        if len(visc_15) == 0 or len(dens_15) == 0:
            continue
        for oil_id in visc_15['oil_id'].unique():
            eta = visc_15[visc_15['oil_id'] == oil_id]['value'].values
            rho = dens_15[dens_15['oil_id'] == oil_id]['value'].values
            if len(eta) > 0 and len(rho) > 0 and rho[0] > 0:
                nu = float(eta[0]) / float(rho[0])  # mPa·s / (g/mL) = mm²/s (cSt)
                kin_records.append((
                    int(oil_id), stage, 'kinematic_viscosity',
                    15.0, 'newtonian', nu, 'mm²/s',
                    None, None, 'computed'
                ))
    
    if kin_records:
        with get_conn() as conn:
            conn.execute("DELETE FROM sample_properties WHERE property_name='kinematic_viscosity'")
            conn.executemany(
                '''INSERT OR IGNORE INTO sample_properties
                   (oil_id, stage_code, property_name, temperature_c, condition,
                    value, unit, std_dev, replicates, method)
                   VALUES (?,?,?,?,?,?,?,?,?,?)''',
                kin_records
            )
        print(f'  ✓ {len(kin_records)} kinematic viscosity records persisted')

    # ── Pseudoplasticity index ──
    # Ratio η(shear_0.1) / η(shear_100) — values >1 = shear-thinning
    pseudo_records = []
    for stage in STAGES:
        s01 = df_visc[(df_visc['stage_code'] == stage) & (df_visc['condition'] == 'shear_0.1')]
        s100 = df_visc[(df_visc['stage_code'] == stage) & (df_visc['condition'] == 'shear_100')]
        for oil_id in set(s01['oil_id']) & set(s100['oil_id']):
            v01 = s01[s01['oil_id'] == oil_id]['value'].values
            v100 = s100[s100['oil_id'] == oil_id]['value'].values
            if len(v01) > 0 and len(v100) > 0 and v100[0] > 0:
                pi = float(v01[0]) / float(v100[0])
                pseudo_records.append((
                    int(oil_id), stage, 'pseudoplasticity_index',
                    None, None, pi, 'ratio',
                    None, None, 'computed'
                ))
    
    if pseudo_records:
        with get_conn() as conn:
            conn.execute("DELETE FROM sample_properties WHERE property_name='pseudoplasticity_index'")
            conn.executemany(
                '''INSERT OR IGNORE INTO sample_properties
                   (oil_id, stage_code, property_name, temperature_c, condition,
                    value, unit, std_dev, replicates, method)
                   VALUES (?,?,?,?,?,?,?,?,?,?)''',
                pseudo_records
            )
        print(f'  ✓ {len(pseudo_records)} pseudoplasticity index records persisted')
    else:
        print('⚠️  pseudoplasticity_index: 0 records — conditions '
              '"shear_0.1" / "shear_100" not found in ECCC ESTS dataset. '
              'Property excluded from ML feature set.')
else:
    print('⚠️ No viscosity data found')


### 6b. Viscosity figures

Viscosity evolution with weathering, Ea distribution, and Ea × API scatter.


In [ ]:
if len(df_visc) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Left: η at 15°C by stage (Newtonian only)
    ax = axes[0]
    newt_15 = df_visc[(df_visc['condition'] == 'newtonian') & (df_visc['temperature_c'] == 15.0)]
    if len(newt_15) > 0:
        data = [np.log10(newt_15[newt_15['stage_code'] == s]['value'].dropna().clip(lower=0.1))
                for s in STAGES]
        bp = ax.boxplot(data, tick_labels=STAGES, patch_artist=True)
        for patch, stage in zip(bp['boxes'], STAGES):
            patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
            patch.set_alpha(0.6)
        ax.set_ylabel('log₁₀(Viscosity) (mPa·s)')
        ax.set_title('Newtonian Viscosity at 15°C')
    
    # Center: Ea histogram
    ax = axes[1]
    if len(df_ea) > 0:
        ax.hist(df_ea['Ea_kJmol'], bins=15, edgecolor='black', alpha=0.7, color='#4477AA')
        ax.axvline(df_ea['Ea_kJmol'].median(), color='red', linestyle='--',
                   label=f'Median={df_ea["Ea_kJmol"].median():.1f}')
        ax.set_xlabel('Activation Energy Ea (kJ/mol)')
        ax.set_ylabel('Count')
        ax.set_title('Arrhenius Ea Distribution')
        ax.legend()
    
    # Right: Ea vs API
    ax = axes[2]
    if len(df_ea) > 0:
        ea_merged = df_ea.merge(df_incl[['oil_id', 'api_gravity', 'oil_type']], on='oil_id')
        valid = ea_merged.dropna(subset=['api_gravity'])
        if len(valid) > 5:
            for otype in sorted(valid['oil_type'].unique()):
                sub = valid[valid['oil_type'] == otype]
                ax.scatter(sub['api_gravity'], sub['Ea_kJmol'], s=40, alpha=0.7,
                           color=OILTYPE_COLORS.get(otype, 'grey'), label=otype)
            r, p = stats.pearsonr(valid['api_gravity'], valid['Ea_kJmol'])
            ax.set_xlabel('API Gravity')
            ax.set_ylabel('Ea (kJ/mol)')
            ax.set_title(f'Ea vs API (r={r:.2f}, p={p:.2e})')
            ax.legend(fontsize=7)
    
    plt.tight_layout()
    fig_path = FIG_DIR / 'nb03_viscosity.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))


## 7. Pour point, flash point, and vapor pressure

Three threshold properties that define operational limits:
- **Pour point** (ASTM D97/D5949): temperature below which oil ceases to flow.
  Increases with weathering (volatiles that depress pour point evaporate).
- **Flash point** (ASTM D7094): minimum temperature for ignition. Increases
  with weathering (flammable volatiles evaporate).
- **Vapor pressure** (ASTM D6377): at 37.8°C. Decreases with weathering.


In [ ]:
with get_conn() as conn:
    df_pour = load_prop_by_stage(conn, 'pour_point')
    df_flash = load_prop_by_stage(conn, 'flash_point')
    df_vap = load_prop_by_stage(conn, 'vapor_pressure', temperature_c=37.8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, df, title, unit in zip(
    axes,
    [df_pour, df_flash, df_vap],
    ['Pour Point', 'Flash Point', 'Vapor Pressure'],
    ['°C', '°C', 'kPa']
):
    if len(df) > 0:
        data = [df[df['stage_code'] == s]['value'].dropna() for s in STAGES]
        bp = ax.boxplot(data, tick_labels=STAGES, patch_artist=True)
        for patch, stage in zip(bp['boxes'], STAGES):
            patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
            patch.set_alpha(0.6)
        ax.set_ylabel(f'{title} ({unit})')
        ax.set_title(f'{title} by Stage (n={df["oil_id"].nunique()} oils)')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)

plt.tight_layout()
fig_path = FIG_DIR / 'nb03_pour_flash_vapor.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# Cross-property: pour point vs flash point (should correlate)
if len(df_pour) > 0 and len(df_flash) > 0:
    w0_pour = df_pour[df_pour['stage_code'] == 'W0'].set_index('oil_id')['value']
    w0_flash = df_flash[df_flash['stage_code'] == 'W0'].set_index('oil_id')['value']
    common = w0_pour.index.intersection(w0_flash.index)
    if len(common) > 5:
        r, p = stats.pearsonr(w0_pour.loc[common], w0_flash.loc[common])
        print(f'Pour vs Flash (W0): r={r:.3f}, p={p:.2e}, n={len(common)}')


## 8. Surface and interfacial tension — spreading coefficient

Measured at oil-air, oil-freshwater, and oil-seawater (3.3% NaCl) interfaces
(ASTM D971 modified / KSV CAM 200 pendant drop, method 12.12/x.x/M).

**Spreading coefficient** S = σ(water-air) − σ(oil-air) − σ(oil-water).
If S > 0, oil spreads spontaneously on water. If S < 0, oil forms lenses.
σ(water-air) ≈ 72.8 mN/m at 15°C (σ(seawater) ≈ 73.5 mN/m).


In [ ]:
with get_conn() as conn:
    # Adaptive tension loading: use names from inventory
    # Try standard names first, fall back to pattern search
    tension_names = {
        'surface': None,
        'freshwater': None,
        'seawater': None,
    }
    for pname in PROP_NAMES:
        plow = pname.lower()
        if 'surface_tension' in plow and 'interfacial' not in plow:
            tension_names['surface'] = pname
        elif 'interfacial' in plow and 'freshwater' in plow:
            tension_names['freshwater'] = pname
        elif 'interfacial' in plow and ('seawater' in plow or 'saline' in plow or 'brine' in plow):
            tension_names['seawater'] = pname
    
    print('Tension property names found:')
    for key, name in tension_names.items():
        print(f'  {key:12s}: {name or "NOT FOUND"}')
    
    df_st = load_prop_by_stage(conn, tension_names['surface'], temperature_c=15.0) if tension_names['surface'] else pd.DataFrame()
    df_it_fw = load_prop_by_stage(conn, tension_names['freshwater'], temperature_c=15.0) if tension_names['freshwater'] else pd.DataFrame()
    df_it_sw = load_prop_by_stage(conn, tension_names['seawater'], temperature_c=15.0) if tension_names['seawater'] else pd.DataFrame()

SIGMA_WATER_AIR = 72.8   # mN/m at 15°C
SIGMA_SEAWATER_AIR = 73.5  # mN/m at 15°C

spread_records = []
if len(df_st) > 0 and len(df_it_sw) > 0:
    for stage in STAGES:
        st = df_st[df_st['stage_code'] == stage].set_index('oil_id')['value']
        it_sw = df_it_sw[df_it_sw['stage_code'] == stage].set_index('oil_id')['value']
        common = st.index.intersection(it_sw.index)
        for oil_id in common:
            S = SIGMA_SEAWATER_AIR - st.loc[oil_id] - it_sw.loc[oil_id]
            spread_records.append((
                int(oil_id), stage, 'spreading_coefficient_seawater',
                15.0, None, float(S), 'mN/m', None, None, 'computed'
            ))
    
    with get_conn() as conn:
        # I1: DELETE before re-insert for computed properties
        conn.execute("DELETE FROM sample_properties WHERE property_name='spreading_coefficient_seawater'")
        conn.executemany(
            '''INSERT OR IGNORE INTO sample_properties
               (oil_id, stage_code, property_name, temperature_c, condition,
                value, unit, std_dev, replicates, method)
               VALUES (?,?,?,?,?,?,?,?,?,?)''',
            spread_records
        )
    
    n_pos = sum(1 for r in spread_records if r[5] > 0)
    print(f'\n✓ {len(spread_records)} spreading coefficients persisted')
    print(f'  S > 0 (spreads): {n_pos}')
    print(f'  S < 0 (lenses):  {len(spread_records) - n_pos}')
else:
    missing = [k for k, v in tension_names.items() if v is None]
    print(f'⚠️ Cannot compute spreading coefficient. Missing: {missing}')


## 9. Water content (Karl Fischer)

Mass percentage of water in the oil (ASTM E203). Fresh oils should have
negligible water. Weathered oils may incorporate water through emulsification.


In [ ]:
with get_conn() as conn:
    df_water = load_prop_by_stage(conn, 'water_content')

if len(df_water) > 0:
    print(f'Water content: {len(df_water)} records ({df_water["oil_id"].nunique()} oils)')
    for stage in STAGES:
        sub = df_water[df_water['stage_code'] == stage]['value'].dropna()
        if len(sub) > 0:
            print(f'  {stage}: median={sub.median():.2f}%, max={sub.max():.2f}% (n={len(sub)})')
else:
    print('⚠️ No water content data')


    # Boxplot if data available
    if len(df_water) > 10:
        fig, ax = plt.subplots(figsize=(6, 4))
        data = [df_water[df_water['stage_code'] == s]['value'].dropna() for s in STAGES]
        bp = ax.boxplot(data, tick_labels=STAGES, patch_artist=True)
        for patch, stage in zip(bp['boxes'], STAGES):
            patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
            patch.set_alpha(0.6)
        ax.set_ylabel('Water Content (%)')
        ax.set_title('Water Content by Weathering Stage')
        fig_path = FIG_DIR / 'nb03_water_content.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close('all')
        display(Image(filename=str(fig_path)))


## 10. Adhesion

Mass of oil per unit area (g/m²) remaining on stainless steel after 30 min
drainage (method 12.14/x.x/M). Increases with weathering as viscosity rises.
Operationally critical: determines cleanup difficulty.


In [ ]:
with get_conn() as conn:
    df_adh = load_prop_by_stage(conn, 'adhesion')

if len(df_adh) > 0:
    print(f'Adhesion: {len(df_adh)} records ({df_adh["oil_id"].nunique()} oils)')
    for stage in STAGES:
        sub = df_adh[df_adh['stage_code'] == stage]['value'].dropna()
        if len(sub) > 0:
            print(f'  {stage}: median={sub.median():.1f} g/m², range=[{sub.min():.1f} – {sub.max():.1f}]')
else:
    print('⚠️ No adhesion data')


    # Boxplot
    if len(df_adh) > 10:
        fig, ax = plt.subplots(figsize=(6, 4))
        data = [df_adh[df_adh['stage_code'] == s]['value'].dropna() for s in STAGES]
        bp = ax.boxplot(data, tick_labels=STAGES, patch_artist=True)
        for patch, stage in zip(bp['boxes'], STAGES):
            patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
            patch.set_alpha(0.6)
        ax.set_ylabel('Adhesion (g/m²)')
        ax.set_title('Adhesion by Weathering Stage')
        fig_path = FIG_DIR / 'nb03_adhesion.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close('all')
        display(Image(filename=str(fig_path)))


## 11. Weathering stage heterogeneity

The stages W0-W3 are nominal — each oil loses a different mass percentage.
W1 may mean 5% loss (heavy crude) or 25% (condensate). This section
quantifies the heterogeneity and assesses risk R17.


In [ ]:
with get_conn() as conn:
    df_ml = load_prop_by_stage(conn, 'evaporative_mass_loss')

if len(df_ml) > 0:
    stats_by_stage = df_ml.groupby('stage_code')['value'].agg(
        ['mean', 'median', 'std', 'min', 'max', 'count']
    )
    stats_by_stage['cv_pct'] = stats_by_stage['std'] / stats_by_stage['mean'] * 100
    print('Mass loss (%) by weathering stage:')
    print(stats_by_stage.round(1).to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Violin
    ax = axes[0]
    data = [df_ml[df_ml['stage_code'] == s]['value'].dropna() for s in STAGES]
    parts = ax.violinplot(data, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(STAGE_COLORS.get(STAGES[i], 'grey'))
        pc.set_alpha(0.6)
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(STAGES)
    ax.set_ylabel('Mass Loss (%)')
    ax.set_title('Stage Heterogeneity')
    
    # Per-oil trajectories
    ax = axes[1]
    for oil_id in df_ml['oil_id'].unique():
        oil_data = df_ml[df_ml['oil_id'] == oil_id].groupby('stage_code')['value'].mean()
        oil_type = df_ml[df_ml['oil_id'] == oil_id]['oil_type'].values[0]
        vals = [oil_data.get(s, np.nan) for s in STAGES]
        ax.plot(range(4), vals, alpha=0.3, linewidth=0.8,
                color=OILTYPE_COLORS.get(oil_type, 'grey'))
    ax.set_xticks(range(4))
    ax.set_xticklabels(STAGES)
    ax.set_ylabel('Mass Loss (%)')
    ax.set_title('Per-Oil Mass Loss Trajectories')
    
    plt.tight_layout()
    fig_path = FIG_DIR / 'nb03_stage_heterogeneity.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    
    max_cv = stats_by_stage['cv_pct'].max()
    print(f'\nR17: max stage CV = {max_cv:.0f}%', end=' → ')
    print('MODERATE' if max_cv < 50 else 'SEVERE', 'limitation')


## §11b. Individual weathering trajectories W0→W3

Per-oil line plots for key physical properties across weathering stages.
Reveals non-monotonic behaviour, abrupt transitions, and outlier oils.

In [ ]:
# Reload key properties (safe: avoids namespace issues between cells)
with get_conn() as conn:
    df_dens_traj = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code, sp.value
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'density' AND sp.temperature_c = 15.0
          AND o.include_in_analysis = 1 AND sp.stage_code IN ('W0','W1','W2','W3')
    ''', conn)
    df_visc_traj = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code, sp.value
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'dynamic_viscosity'
          AND sp.temperature_c = 15.0 AND sp.condition = 'newtonian'
          AND o.include_in_analysis = 1 AND sp.stage_code IN ('W0','W1','W2','W3')
    ''', conn)
    df_pour_traj = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code, sp.value
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'pour_point'
          AND o.include_in_analysis = 1 AND sp.stage_code IN ('W0','W1','W2','W3')
    ''', conn)
    df_flash_traj = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code, sp.value
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'flash_point'
          AND o.include_in_analysis = 1 AND sp.stage_code IN ('W0','W1','W2','W3')
    ''', conn)

datasets = [
    (df_dens_traj,  'Density at 15°C (g/mL)',   False),
    (df_visc_traj,  'Viscosity at 15°C (mPa·s)', True),
    (df_pour_traj,  'Pour Point (°C)',            False),
    (df_flash_traj, 'Flash Point (°C)',           False),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
stage_x = {'W0': 0, 'W1': 1, 'W2': 2, 'W3': 3}

for ax, (df_prop, ylabel, use_log) in zip(axes.flat, datasets):
    if len(df_prop) == 0:
        ax.text(0.5, 0.5, f'{ylabel}\nNo data', ha='center', va='center',
                transform=ax.transAxes)
        continue
    for oil_id in df_prop['oil_id'].unique():
        sub = df_prop[df_prop['oil_id'] == oil_id]
        otype = sub['oil_type'].iloc[0]
        # Aggregate duplicates
        pts = sub.groupby('stage_code')['value'].mean()
        xs = [stage_x[s] for s in pts.index]
        ys = pts.values
        ax.plot(xs, ys, alpha=0.35, linewidth=0.9,
                color=OILTYPE_COLORS.get(otype, 'grey'))
    # Legend proxies
    for otype in sorted(df_prop['oil_type'].unique()):
        ax.plot([], [], color=OILTYPE_COLORS.get(otype, 'grey'), linewidth=2, label=otype)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(STAGES)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    if use_log:
        ax.set_yscale('log')
    ax.legend(fontsize=7)

fig.suptitle('Individual Weathering Trajectories W0→W3', fontsize=13, fontweight='bold')
plt.tight_layout()
fig_path = FIG_DIR / 'nb03_trajectories.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print(f'Saved: {fig_path}')


## §11c. Thickening factor η(W3)/η(W0)

Ratio of Newtonian viscosity at 15°C between W3 (most weathered) and W0 (fresh).
Characterises the rheological sensitivity of each oil to evaporative weathering.

In [ ]:
# Compute thickening factor from Newtonian viscosity at 15°C
visc_newt_15 = df_visc_traj.groupby(['oil_id', 'stage_code'])['value'].mean().unstack('stage_code')

if 'W0' in visc_newt_15.columns and 'W3' in visc_newt_15.columns:
    valid = visc_newt_15.dropna(subset=['W0', 'W3'])
    valid = valid[valid['W0'] > 0]
    thickening = (valid['W3'] / valid['W0']).rename('thickening_factor')

    # Merge oil metadata
    df_thick = thickening.reset_index().merge(
        df_incl[['oil_id', 'oil_name', 'oil_type', 'api_gravity']], on='oil_id'
    )

    print(f'Thickening factor computed for {len(df_thick)} oils')
    print(f'  Median: {df_thick["thickening_factor"].median():.1f}x')
    print(f'  Range:  {df_thick["thickening_factor"].min():.1f}x – '
          f'{df_thick["thickening_factor"].max():.0f}x')
    print()
    print('By oil_type (median thickening):')
    for otype, grp in df_thick.groupby('oil_type'):
        print(f'  {otype:20s}: {grp["thickening_factor"].median():.1f}x '
              f'(n={len(grp)}, max={grp["thickening_factor"].max():.0f}x)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: histogram by oil_type
    ax = axes[0]
    for otype in sorted(df_thick['oil_type'].unique()):
        sub = df_thick[df_thick['oil_type'] == otype]['thickening_factor']
        ax.hist(np.log10(sub.clip(lower=0.1)), bins=12, alpha=0.5,
                label=f'{otype} (n={len(sub)})',
                color=OILTYPE_COLORS.get(otype, 'grey'))
    ax.set_xlabel('log10(Thickening Factor)')
    ax.set_ylabel('Count')
    ax.set_title('Viscosity Thickening Factor Distribution')
    ax.axvline(np.log10(10), color='grey', ls=':', alpha=0.6)
    ax.axvline(np.log10(100), color='grey', ls=':', alpha=0.6)
    ax.text(np.log10(10)+0.05, ax.get_ylim()[1]*0.9, '10x', fontsize=8, color='grey')
    ax.text(np.log10(100)+0.05, ax.get_ylim()[1]*0.9, '100x', fontsize=8, color='grey')
    ax.legend(fontsize=7)

    # Right: thickening vs API gravity
    ax = axes[1]
    valid_api = df_thick.dropna(subset=['api_gravity'])
    for otype in sorted(valid_api['oil_type'].unique()):
        sub = valid_api[valid_api['oil_type'] == otype]
        ax.scatter(sub['api_gravity'], np.log10(sub['thickening_factor']),
                   s=40, alpha=0.7, color=OILTYPE_COLORS.get(otype, 'grey'), label=otype)
    if len(valid_api) > 5:
        r, p = stats.pearsonr(valid_api['api_gravity'],
                              np.log10(valid_api['thickening_factor']))
        ax.set_title(f'Thickening vs API Gravity (r={r:.2f}, p={p:.3f})')
    ax.set_xlabel('API Gravity')
    ax.set_ylabel('log10(Thickening Factor)')
    ax.legend(fontsize=7)

    plt.tight_layout()
    fig_path = FIG_DIR / 'nb03_thickening.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))

    # Persist thickening factor
    thick_records = [
        (int(row['oil_id']), 'W0', 'thickening_factor_15C',
         15.0, 'newtonian', float(row['thickening_factor']),
         'ratio', None, None, 'computed')
        for _, row in df_thick.iterrows()
    ]
    with get_conn() as conn:
        conn.execute("DELETE FROM sample_properties WHERE property_name='thickening_factor_15C'")
        conn.executemany(
            '''INSERT OR IGNORE INTO sample_properties
               (oil_id, stage_code, property_name, temperature_c, condition,
                value, unit, std_dev, replicates, method)
               VALUES (?,?,?,?,?,?,?,?,?,?)''',
            thick_records
        )
    print(f'\n Persisted {len(thick_records)} thickening_factor_15C records')
else:
    print(' Insufficient W0/W3 viscosity data for thickening factor')


## §11d. Weathering sensitivity: Δprop / Δmass_loss

Normalises the property change (W3−W0) by the evaporated mass fraction at W3.
Separates oils that changed because they lost more mass from those intrinsically sensitive.

In [ ]:
with get_conn() as conn:
    df_ml_sens = pd.read_sql('''
        SELECT sp.oil_id, sp.stage_code, sp.value as mass_loss
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'evaporative_mass_loss'
          AND o.include_in_analysis = 1
    ''', conn)

w3_ml = (df_ml_sens[df_ml_sens['stage_code'] == 'W3']
         .groupby('oil_id')['mass_loss'].mean())

# Density sensitivity
w0_d = df_dens_traj[df_dens_traj['stage_code']=='W0'].groupby('oil_id')['value'].mean()
w3_d = df_dens_traj[df_dens_traj['stage_code']=='W3'].groupby('oil_id')['value'].mean()
common_d = w0_d.index.intersection(w3_d.index).intersection(w3_ml.index)
common_d = common_d[w3_ml.loc[common_d] > 0]
sens_dens = ((w3_d.loc[common_d] - w0_d.loc[common_d]) /
             w3_ml.loc[common_d] * 100).rename('sens_density')

# Viscosity sensitivity (log scale: log(W3) - log(W0))
w0_v = df_visc_traj[df_visc_traj['stage_code']=='W0'].groupby('oil_id')['value'].mean()
w3_v = df_visc_traj[df_visc_traj['stage_code']=='W3'].groupby('oil_id')['value'].mean()
common_v = w0_v.index.intersection(w3_v.index).intersection(w3_ml.index)
common_v = common_v[(w3_ml.loc[common_v] > 0) & (w0_v.loc[common_v] > 0)]
sens_visc = ((np.log10(w3_v.loc[common_v]) - np.log10(w0_v.loc[common_v])) /
             w3_ml.loc[common_v] * 100).rename('sens_log_viscosity')

print(f'Density sensitivity: {len(sens_dens)} oils')
print(f'  mean={sens_dens.mean():.4f} g/mL per % mass loss')
print(f'  range=[{sens_dens.min():.4f}, {sens_dens.max():.4f}]')
print(f'Viscosity sensitivity (log10): {len(sens_visc)} oils')
print(f'  mean={sens_visc.mean():.4f} log10(mPa·s) per % mass loss')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: density sensitivity by oil_type
ax = axes[0]
df_sens_d = sens_dens.reset_index().merge(
    df_incl[['oil_id', 'oil_type']], on='oil_id')
for otype in sorted(df_sens_d['oil_type'].unique()):
    sub = df_sens_d[df_sens_d['oil_type'] == otype]['sens_density']
    ax.scatter([otype]*len(sub), sub, alpha=0.6,
               color=OILTYPE_COLORS.get(otype, 'grey'), s=40)
ax.axhline(0, color='grey', ls='--', alpha=0.4)
ax.set_xlabel('Oil Type')
ax.set_ylabel('Density sensitivity (g/mL per % mass loss)')
ax.set_title('Density Weathering Sensitivity by Oil Type')
ax.tick_params(axis='x', rotation=15)

# Right: viscosity sensitivity vs API
ax = axes[1]
df_sens_v = sens_visc.reset_index().merge(
    df_incl[['oil_id', 'oil_type', 'api_gravity']], on='oil_id')
valid_api = df_sens_v.dropna(subset=['api_gravity'])
for otype in sorted(valid_api['oil_type'].unique()):
    sub = valid_api[valid_api['oil_type'] == otype]
    ax.scatter(sub['api_gravity'], sub['sens_log_viscosity'],
               s=40, alpha=0.7, color=OILTYPE_COLORS.get(otype, 'grey'), label=otype)
if len(valid_api) > 5:
    r, p = stats.pearsonr(valid_api['api_gravity'], valid_api['sens_log_viscosity'])
    ax.set_title(f'Viscosity Sensitivity vs API (r={r:.2f}, p={p:.3f})')
ax.set_xlabel('API Gravity')
ax.set_ylabel('log10(Viscosity) sensitivity per % mass loss')
ax.legend(fontsize=7)

plt.tight_layout()
fig_path = FIG_DIR / 'nb03_sensitivity.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


## §11e. Spearman correlation heatmap (W0 physical properties)

Full correlation matrix across all physical properties with adequate coverage (n > 20 oils).
Spearman used: robust to log-normal distributions (viscosity) and monotonic non-linear relations.

In [ ]:
with get_conn() as conn:
    df_water_corr = pd.read_sql('''
        SELECT sp.oil_id, sp.value FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'water_content'
          AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
    ''', conn)
    df_adh_corr = pd.read_sql('''
        SELECT sp.oil_id, sp.value FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'adhesion'
          AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
    ''', conn)
    df_spread_corr = pd.read_sql('''
        SELECT sp.oil_id, sp.value FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'spreading_coefficient_seawater'
          AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
    ''', conn)

# Build wide DataFrame at W0
oil_ids = df_incl['oil_id'].values
wide = pd.DataFrame(index=oil_ids)
wide.index.name = 'oil_id'

# From sample_properties (W0, included oils)
def w0_mean(df):
    return df.groupby('oil_id')['value'].mean()

wide['density_15C']   = w0_mean(df_dens_traj[df_dens_traj['stage_code']=='W0'])
wide['viscosity_15C'] = np.log10(w0_mean(df_visc_traj[df_visc_traj['stage_code']=='W0']).clip(lower=0.01))
wide['pour_point']    = w0_mean(df_pour_traj[df_pour_traj['stage_code']=='W0'])
wide['flash_point']   = w0_mean(df_flash_traj[df_flash_traj['stage_code']=='W0'])
wide['water_content'] = w0_mean(df_water_corr)
wide['adhesion']      = w0_mean(df_adh_corr)
wide['spreading_coef']= w0_mean(df_spread_corr)

# From oils table
oil_meta = df_incl.set_index('oil_id')
wide['api_gravity'] = oil_meta['api_gravity']
wide['sulfur_pct']  = oil_meta['sulfur_pct']
wide['wax_pct']     = oil_meta['wax_pct']

# Keep only columns with >= 20 non-null
wide = wide.loc[:, wide.notna().sum() >= 20]
print(f'Wide matrix: {wide.shape[0]} oils x {wide.shape[1]} properties')
print('Coverage per property:')
for col in wide.columns:
    print(f'  {col:20s}: {wide[col].notna().sum()} oils')

corr = wide.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman r')
labels = corr.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
# Annotate cells
for i in range(len(labels)):
    for j in range(len(labels)):
        v = corr.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if abs(v) > 0.6 else 'black')
ax.set_title('Spearman Correlation — W0 Physical Properties')
fig_path = FIG_DIR / 'nb03_property_correlation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


## §11f. Kruskal-Wallis test by oil_type (W0)

Non-parametric ANOVA across oil_type groups. Quantifies the oil_type confound (R20).
Justifies crude-only configurations (C8) in the ML pipeline.

In [ ]:
from scipy.stats import kruskal
import numpy as np

# ── Rebuild 'wide' if not in namespace (§11f is self-contained) ──
if 'wide' not in dir():
    with get_conn() as conn:
        df_dens_traj_ = pd.read_sql('''
            SELECT sp.oil_id, sp.stage_code, sp.value
            FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'density' AND sp.temperature_c = 15.0
              AND o.include_in_analysis = 1 AND sp.stage_code = 'W0'
        ''', conn)
        df_visc_traj_ = pd.read_sql('''
            SELECT sp.oil_id, sp.stage_code, sp.value
            FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'dynamic_viscosity'
              AND sp.temperature_c = 15.0 AND sp.condition = 'newtonian'
              AND o.include_in_analysis = 1 AND sp.stage_code = 'W0'
        ''', conn)
        df_pour_traj_ = pd.read_sql('''
            SELECT sp.oil_id, sp.stage_code, sp.value
            FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'pour_point'
              AND o.include_in_analysis = 1 AND sp.stage_code = 'W0'
        ''', conn)
        df_flash_traj_ = pd.read_sql('''
            SELECT sp.oil_id, sp.stage_code, sp.value
            FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'flash_point'
              AND o.include_in_analysis = 1 AND sp.stage_code = 'W0'
        ''', conn)
        df_water_corr_ = pd.read_sql('''
            SELECT sp.oil_id, sp.value FROM sample_properties sp
            JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'water_content'
              AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
        ''', conn)
        df_adh_corr_ = pd.read_sql('''
            SELECT sp.oil_id, sp.value FROM sample_properties sp
            JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'adhesion'
              AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
        ''', conn)
        df_spread_corr_ = pd.read_sql('''
            SELECT sp.oil_id, sp.value FROM sample_properties sp
            JOIN oils o ON sp.oil_id = o.oil_id
            WHERE sp.property_name = 'spreading_coefficient_seawater'
              AND sp.stage_code = 'W0' AND o.include_in_analysis = 1
        ''', conn)

    def _w0_mean(df):
        return df.groupby('oil_id')['value'].mean()

    oil_ids = df_incl['oil_id'].values
    wide = pd.DataFrame(index=oil_ids)
    wide.index.name = 'oil_id'
    wide['density_15C']    = _w0_mean(df_dens_traj_)
    wide['viscosity_15C']  = np.log10(_w0_mean(df_visc_traj_).clip(lower=0.01))
    wide['pour_point']     = _w0_mean(df_pour_traj_)
    wide['flash_point']    = _w0_mean(df_flash_traj_)
    wide['water_content']  = _w0_mean(df_water_corr_)
    wide['adhesion']       = _w0_mean(df_adh_corr_)
    wide['spreading_coef'] = _w0_mean(df_spread_corr_)
    oil_meta = df_incl.set_index('oil_id')
    wide['api_gravity'] = oil_meta['api_gravity']
    wide['sulfur_pct']  = oil_meta['sulfur_pct']
    wide['wax_pct']     = oil_meta['wax_pct']
    wide = wide.loc[:, wide.notna().sum() >= 20]
    print(f'Wide rebuilt: {wide.shape[0]} oils x {wide.shape[1]} properties')

kw_results = []
for col in wide.columns:
    groups, group_labels = [], []
    for otype, grp in df_incl.groupby('oil_type'):
        ids = grp['oil_id'].values
        vals = wide.loc[wide.index.isin(ids), col].dropna().values
        if len(vals) >= 3:
            groups.append(vals)
            group_labels.append(otype)

    if len(groups) >= 2:
        H, p = kruskal(*groups)
        n_total = sum(len(g) for g in groups)
        k = len(groups)
        eta_sq = (H - k + 1) / (n_total - k) if (n_total - k) > 0 else np.nan
        kw_results.append({
            'property': col, 'H': round(H, 2), 'p': p,
            'eta_sq': round(eta_sq, 3) if not np.isnan(eta_sq) else np.nan,
            'n_groups': k, 'n_total': n_total,
            'significant': p < 0.05
        })

df_kw = pd.DataFrame(kw_results).sort_values('p')
print('Kruskal-Wallis test: physical properties vs oil_type (W0)')
print(f'{"Property":22s} {"H":>7} {"p":>10} {"eta_sq":>8} {"sig":>5}')
print('-' * 60)
for _, row in df_kw.iterrows():
    sig = '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else
          ('*' if row['p'] < 0.05 else 'ns'))
    print(f'{row["property"]:22s} {row["H"]:7.2f} {row["p"]:10.4f} '
          f'{row["eta_sq"]:8.3f} {sig:>5}')

fig, ax = plt.subplots(figsize=(10, 5))
props = df_kw['property'].tolist()
neg_log_p = [-np.log10(p) if p > 0 else 20 for p in df_kw['p']]
colors = ['#CC3333' if s else '#88AACC' for s in df_kw['significant']]
ax.barh(range(len(props)), neg_log_p, color=colors, alpha=0.8)
ax.axvline(-np.log10(0.05), color='grey', ls='--', alpha=0.7, label='p=0.05')
ax.set_yticks(range(len(props)))
ax.set_yticklabels(props, fontsize=9)
ax.set_xlabel('-log10(p-value)')
ax.set_title('Kruskal-Wallis: Oil Type Effect on Physical Properties (W0)')
ax.legend()
plt.tight_layout()
fig_path = FIG_DIR / 'nb03_kruskal_wallis.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


## 12. Outlier detection (3σ per property)

Statistical outliers per physical property. Listed for human review —
not automatically removed. Outlier treatment decisions belong in NB03f.


In [ ]:
with get_conn() as conn:
    df_all_props = pd.read_sql('''
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.stage_code,
               sp.property_name, sp.value
        FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE o.include_in_analysis = 1
          AND sp.value IS NOT NULL
          AND sp.method != 'computed'
    ''', conn)

# Outlier scan per property
outlier_inventory = []
for prop in df_all_props['property_name'].unique():
    sub = df_all_props[df_all_props['property_name'] == prop]['value']
    if len(sub) < 10:
        continue
    # Use log-scale for viscosity (log-normal distribution)
    if 'viscosity' in prop.lower() and (sub > 0).all():
        sub = np.log10(sub)
    mean, std = sub.mean(), sub.std()
    if std == 0:
        continue
    mask = (sub - mean).abs() > 3 * std
    if mask.sum() > 0:
        outliers = df_all_props[(df_all_props['property_name'] == prop) & 
                                 (df_all_props['value'] - mean).abs() > 3 * std]
        for _, row in outliers.iterrows():
            z = (row['value'] - mean) / std
            outlier_inventory.append({
                'property': prop, 'oil_name': row['oil_name'],
                'stage': row['stage_code'], 'value': row['value'],
                'z_score': z
            })

df_outliers = pd.DataFrame(outlier_inventory)
if len(df_outliers) > 0:
    print(f'Outlier inventory: {len(df_outliers)} values >3σ across {df_outliers["property"].nunique()} properties')
    for prop in df_outliers['property'].unique():
        sub = df_outliers[df_outliers['property'] == prop]
        print(f'\n  {prop}: {len(sub)} outliers')
        for _, row in sub.head(5).iterrows():
            print(f'    {row["oil_name"]} ({row["stage"]}): {row["value"]:.3g} (z={row["z_score"]:+.1f})')
else:
    print('✓ No outliers >3σ detected')

# Save for NB03f
if len(df_outliers) > 0:
    csv_path = FIG_DIR / 'nb03_outlier_inventory.csv'
    df_outliers.to_csv(csv_path, index=False)
    print(f'\nSaved: {csv_path}')


## 13. Data richness heatmap

Completeness per oil × property section. Identifies oils with sparse data
(which may behave anomalously in ML) and property sections with systematic gaps.


In [ ]:
# Group properties into sections
PROP_SECTIONS = {
    'density': lambda p: p.startswith('density'),
    'viscosity': lambda p: p.startswith('dynamic_viscosity'),
    'tension': lambda p: 'tension' in p,
    'pour_flash_vap': lambda p: p in ('pour_point', 'flash_point', 'vapor_pressure'),
    'SARA': lambda p: p.startswith('sara_'),
    'BTEX': lambda p: p.startswith('btex_'),
    'emulsion': lambda p: p.startswith('emulsion_'),
    'adhesion': lambda p: p == 'adhesion',
    'dispersibility': lambda p: p.startswith('dispersant'),
    'PHC': lambda p: p.startswith('phc_'),
    'SIMDIS': lambda p: p.startswith('bp_'),
    'wax_sulfur': lambda p: p in ('wax_content', 'sulfur_content'),
}

with get_conn() as conn:
    # For each oil × section: has any data?
    oil_ids = df_incl['oil_id'].values
    oil_names = df_incl.set_index('oil_id')['oil_name'].to_dict()
    
    all_props = pd.read_sql('''
        SELECT DISTINCT oil_id, property_name
        FROM sample_properties
        WHERE oil_id IN ({}) AND value IS NOT NULL
    '''.format(','.join(str(i) for i in oil_ids)), conn)

richness = pd.DataFrame(0, index=oil_ids, columns=list(PROP_SECTIONS.keys()))
for _, row in all_props.iterrows():
    for section, check_fn in PROP_SECTIONS.items():
        if check_fn(row['property_name']):
            richness.loc[row['oil_id'], section] = 1
            break

richness.index = [oil_names.get(i, str(i))[:25] for i in richness.index]

fig, ax = plt.subplots(figsize=(10, max(8, len(richness) * 0.2)))
im = ax.imshow(richness.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(richness.columns)))
ax.set_xticklabels(richness.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(richness)))
ax.set_yticklabels(richness.index, fontsize=5)
ax.set_title('Data Richness: Oil × Property Section (green=present, red=absent)')
fig_path = FIG_DIR / 'nb03_data_richness.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# Summary
completeness = richness.sum(axis=1) / len(richness.columns) * 100
print(f'Data richness summary:')
print(f'  Rich (>80% sections):     {(completeness > 80).sum()} oils')
print(f'  Moderate (50-80%):         {((completeness >= 50) & (completeness <= 80)).sum()} oils')
print(f'  Sparse (<50%):             {(completeness < 50).sum()} oils')


## 14. Cross-property consistency

Physically expected relationships between properties. Deviations indicate
parsing errors, measurement issues, or genuinely anomalous oils.

Expected: API↔density (inverse), pour↔wax (positive), flash↔vapor pressure
(inverse), density↔viscosity (positive for same T).


In [ ]:
pairs = [
    ('api_gravity', 'density_15c', 'API × Density', -0.9),
    ('pour_point_c', 'wax_pct', 'Pour Point × Wax', 0.5),
    ('api_gravity', 'pour_point_c', 'API × Pour Point', -0.4),
    ('sulfur_pct', 'api_gravity', 'Sulfur × API', -0.3),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (x_col, y_col, title, expected_sign) in zip(axes.flat, pairs):
    valid = df_incl.dropna(subset=[x_col, y_col])
    if len(valid) < 5:
        ax.text(0.5, 0.5, f'{title}\nInsufficient data', ha='center', va='center',
                transform=ax.transAxes)
        continue
    
    for otype in sorted(valid['oil_type'].unique()):
        sub = valid[valid['oil_type'] == otype]
        ax.scatter(sub[x_col], sub[y_col], s=30, alpha=0.7,
                   color=OILTYPE_COLORS.get(otype, 'grey'), label=otype)
    
    r, p = stats.pearsonr(valid[x_col], valid[y_col])
    sign_ok = '✓' if (r * expected_sign > 0) else '⚠️ unexpected sign'
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{title} (r={r:.2f} {sign_ok})')
    ax.legend(fontsize=6)

plt.tight_layout()
fig_path = FIG_DIR / 'nb03_cross_property.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


## 15. Comparison with literature ranges

Validate ECCC ESTS ranges against published reference values.
Sources: NAS 2003, Fingas 2015, ITOPF 2024.


In [ ]:
# Literature ranges (approximate)
LITERATURE = {
    'flash_point_c': {
        'Gasoline': (-43, -38), 'Diesel': (52, 96),
        'Crude (typical)': (-30, 60),
        'source': 'ITOPF 2024'
    },
    'sulfur_pct': {
        'Sweet crude': (0, 0.5), 'Sour crude': (0.5, 5),
        'source': 'Speight 2014'
    },
    'wax_pct': {
        'Low wax': (0, 5), 'Medium wax': (5, 15), 'High wax': (15, 40),
        'source': 'Fingas 2015'
    },
    'api_gravity': {
        'Light crude': (31, 55), 'Medium crude': (22, 31),
        'Heavy crude': (10, 22), 'Extra-heavy/bitumen': (-5, 10),
        'source': 'NAS 2003'
    },
    'density_15c': {
        'Crude (typical)': (0.78, 1.00), 'Refined (diesel)': (0.82, 0.88),
        'Bitumen': (1.00, 1.05),
        'source': 'ASTM D5002'
    },
    'pour_point_c': {
        'Low pour (paraffinic)': (-40, -10), 'High pour (waxy)': (0, 40),
        'source': 'Fingas 2015'
    },
}

print('ECCC ESTS ranges vs literature (W0, included oils):')
print(f'{"Property":25s} {"ECCC range":20s} {"Literature":30s} {"Source"}')
print('-' * 90)

for prop in ['api_gravity', 'density_15c', 'pour_point_c', 'flash_point_c',
             'sulfur_pct', 'wax_pct']:
    vals = df_incl[prop].dropna()
    if len(vals) == 0:
        continue
    eccc_range = f'{vals.min():.1f} – {vals.max():.1f}'
    lit_info = LITERATURE.get(prop, {})
    if lit_info:
        source = lit_info.pop('source', '')
        lit_str = '; '.join(f'{k}: {v[0]}-{v[1]}' for k, v in lit_info.items())
        print(f'{prop:25s} {eccc_range:20s} {lit_str:30s} {source}')
    else:
        print(f'{prop:25s} {eccc_range:20s} {"(no reference)":30s}')


## 16. Summary

Overview of all findings, derived properties, and data quality assessment.


In [ ]:
with get_conn() as conn:
    nb03_props = ['thermal_expansion_coeff', 'viscosity_Ea_kJmol',
                  'kinematic_viscosity', 'pseudoplasticity_index',
                  'spreading_coefficient_seawater']
    placeholders = ','.join('?' * len(nb03_props))
    n_derived = conn.execute(
        f'SELECT COUNT(*) FROM sample_properties WHERE property_name IN ({placeholders})',
        nb03_props
    ).fetchone()[0]
    n_derived_types = conn.execute(
        f'SELECT COUNT(DISTINCT property_name) FROM sample_properties '
        f'WHERE property_name IN ({placeholders})',
        nb03_props
    ).fetchone()[0]

print('=' * 60)
print('NB03 — Physical Properties and Data Quality Summary')
print('=' * 60)
print(f'  Oils analyzed:           {len(df_incl)}')
print(f'  Property types in DB:    {checks["n_property_types"]}')
print(f'  Outliers detected (3σ):  {len(df_outliers) if len(df_outliers) > 0 else 0}')
print(f'  Derived properties:      {n_derived:,} records ({n_derived_types} types)')
print(f'  Figures:                 {len(list(FIG_DIR.glob("*.png")))}')
print()
print('  Derived properties persisted:')
for prop in ['thermal_expansion_coeff', 'viscosity_Ea_kJmol',
             'kinematic_viscosity', 'pseudoplasticity_index',
             'spreading_coefficient_seawater']:
    with get_conn() as conn:
        n = conn.execute(
            'SELECT COUNT(*) FROM sample_properties WHERE property_name=?', (prop,)
        ).fetchone()[0]
    print(f'    {prop:40s}: {n:>5} records')
# Document low-coverage properties excluded from ML
LOW_COVERAGE = ['thermal_expansion_coeff', 'viscosity_Ea_kJmol', 'pseudoplasticity_index']
print('\n  ⚠️  Low-coverage properties (excluded from ML feature set):')
for prop in LOW_COVERAGE:
    with get_conn() as conn:
        n = conn.execute(
            'SELECT COUNT(DISTINCT oil_id) FROM sample_properties WHERE property_name=?',
            (prop,)
        ).fetchone()[0]
    pct = 100 * n / len(df_incl)
    print(f'    {prop:40s}: {n:>3} oils ({pct:.0f}% coverage)')

print(f'\n✓ NB03 complete. Next: NB03b (Bulk Composition).')
